[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C06/E06-2_Identification_Conformite.ipynb)

# Présence d'un défaut sur une pièce : classer par ressemblance et par probabilités

## Le contexte

Une station d'inspection photographie la surface de chaque pièce et en extrait trente descripteurs de l'anomalie détectée. Dix grandeurs de base sont mesurées : le rayon, la variance de texture, le périmètre, l'aire, le lissage, la compacité, la concavité, le nombre de points concaves, la symétrie et la dimension fractale. Pour chacune, la station fournit la valeur moyenne, son erreur de mesure et sa valeur extrême. L'opérateur doit décider si la pièce présente un défaut ou non.

La grandeur à prédire est une catégorie, défaut présent ou pas de défaut : c'est un problème de classification binaire.

Défi : prédire `presence_defaut` et obtenir un F-score macro supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est celui des plus proches voisins, qui classe une pièce comme ses voisins les plus proches, avec ses réglages par défaut sur les descripteurs bruts ; il atteint un F-score macro de 0,924 : à vous de faire mieux. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Identification_Conformite.xlsx`, dans le dossier `donnees` du dépôt : 569 pièces, 32 colonnes. Une ligne est une pièce.

| Colonne | Signification | Unité |
|---|---|---|
| `rayon_moyen_defaut`, `variance_texture`, `perimetre_defaut`, `aire_defaut`, `lissage_defaut`, `compacite_defaut`, `concavite_defaut`, `points_concaves_defaut`, `symetrie_defaut`, `dimension_fractale_defaut` | dix descripteurs moyens de l'anomalie | sans unité déclarée |
| `erreur_..._defaut` | les dix erreurs de mesure associées, une par descripteur | sans unité déclarée |
| `..._max_...` | les dix valeurs extrêmes associées, une par descripteur | sans unité déclarée |
| `indice_severite_normalise` | indice de sévérité construit, examiné à l'étape 3 | sans unité |
| `presence_defaut` | cible : `Defaut_present` ou `Pas_de_defaut` | catégorie |

Origine : jeu public transformé. Les trente descripteurs, aux valeurs inchangées, sont renommés en descripteurs de défaut, la grandeur à prédire devient la présence d'un défaut et un indice de sévérité est construit. Il enseigne une méthode, et aucune conclusion de contrôle n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la répartition des deux réponses avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Identification_Conformite.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Identification_Conformite.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "presence_defaut"
print("dimensions :", df.shape)
print("cases vides au total :", int(df.isna().sum().sum()))
print(df[cible].value_counts().to_string())
df.iloc[:3, :6]

Lecture du résultat : 569 pièces, 32 colonnes, aucune case vide. 212 pièces présentent un défaut et 357 n'en présentent pas : les deux réponses sont assez équilibrées pour que l'exactitude ait un sens, mais le F-score macro les compte pour autant.

### Étape 2 : mettre le test de côté

Un quart des pièces est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Le tirage est stratifié, pour que les deux réponses soient représentées dans les deux bases dans la même proportion.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=GRAINE, stratify=y)
print("apprentissage :", len(X_train), "pièces")
print("test :", len(X_test), "pièces")

### Étape 3 : écarter la colonne construite, regarder les échelles

L'indice de sévérité est construit à partir des autres colonnes. On le vérifie sur l'apprentissage seul : si une régression linéaire des trente descripteurs le retrouve exactement, il n'apporte aucune information de plus et il est écarté. On regarde ensuite l'écart-type de chaque descripteur : un modèle qui compare les pièces par une distance verra surtout les descripteurs qui varient le plus.

In [ ]:
from sklearn.linear_model import LinearRegression

descripteurs = [colonne for colonne in X_train.columns if colonne != "indice_severite_normalise"]
reconstruction = LinearRegression().fit(X_train[descripteurs], X_train["indice_severite_normalise"])
r2_indice = reconstruction.score(X_train[descripteurs], X_train["indice_severite_normalise"])
print(f"R² de l'indice reconstruit par les trente descripteurs : {r2_indice:.4f}")
X_train, X_test = X_train[descripteurs], X_test[descripteurs]

ecarts = X_train.std().sort_values(ascending=False)
print(ecarts.head(3).round(3).to_string())
print(ecarts.tail(3).round(4).to_string())

Lecture du résultat : la régression linéaire retrouve l'indice avec un R² de 1,0000 : c'est une combinaison des trente descripteurs, sans information nouvelle, et il est écarté. Les écarts-types vont de 0,0028 pour `erreur_dimension_fractale_defaut` à 559 pour `aire_max_defaut`, soit un rapport de plusieurs ordres de grandeur. Une distance calculée sur ces valeurs brutes est gouvernée par `aire_max_defaut` et `aire_defaut` ; les autres descripteurs comptent à peine.

### Étape 4 : construire le modèle de départ

Le modèle de départ est celui de la section du livre : les plus proches voisins, qui classent une pièce comme la majorité de ses cinq voisins les plus proches, au sens de la distance entre descripteurs. Il garde ses réglages par défaut et les descripteurs gardent leurs unités brutes. Il apprend sur `X_train` seul.

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle prédit les pièces qu'il n'a jamais vues. On lit le F-score macro, l'exactitude et le F-score de chaque réponse. Un témoin sans apprentissage, qui répond toujours la réponse la plus fréquente, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

y_pred = modele.predict(X_test)
score_depart = f1_score(y_test, y_pred, average="macro")
temoin = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
score_temoin = f1_score(y_test, temoin.predict(X_test), average="macro")
print(f"F-score macro : {score_depart:.3f}")
print(f"exactitude : {accuracy_score(y_test, y_pred):.3f}")
print(f"F-score macro du témoin (toujours la réponse la plus fréquente) : {score_temoin:.3f}")
classes = list(modele.classes_)
print(pd.Series(f1_score(y_test, y_pred, average=None, labels=classes), index=classes).round(3).to_string())

Lecture du résultat : le F-score macro est de 0,924 pour une exactitude de 0,930, nettement au-dessus du témoin (0,386). Le modèle est bon, mais il décide avec des distances dominées par quelques descripteurs : on peut espérer mieux. C'est ce chiffre, mesuré sur des pièces que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : mettre à l'échelle, puis comparer les trois méthodes de la section

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les deux cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première compte les paires de descripteurs presque redondantes sur l'apprentissage, puis compare par validation croisée les plus proches voisins bruts, les plus proches voisins à l'échelle, le Bayes naïf gaussien, l'analyse discriminante linéaire et, pour comparaison, la régression logistique du chapitre précédent. La seconde mesure le modèle retenu une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le F-score macro du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = f1_score(y_test, modele.predict(X_test), average="macro")
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE f1_macro = %.3f" % score)

## Synthèse

1. Les plus proches voisins classent une pièce comme ses voisins les plus proches : tout repose sur la distance, donc sur l'échelle des descripteurs.
2. Sur des valeurs brutes, les descripteurs aux grands nombres écrasent les autres ; la mise à l'échelle, ajustée sur l'apprentissage seul, rétablit l'équilibre.
3. Le Bayes naïf traite chaque descripteur séparément et suppose qu'ils sont indépendants : des descripteurs très corrélés, comme l'aire et le périmètre, contredisent cette hypothèse.
4. L'analyse discriminante linéaire tolère mieux les descripteurs corrélés, au prix d'une frontière plane.
5. Une colonne construite à partir des autres, comme l'indice de sévérité, n'apporte rien et s'écarte ; le choix du modèle se fait par validation croisée sur l'apprentissage, et le test ne sert qu'une fois.